# Lab 7: Label texts with Gemini

**Allow 45 minutes.** Work in a copy named `lab07_call_gemini_from_python_mine.ipynb`.

In the Week 7 interactive you labeled comments with a simulated Gemini. Now you send the calls to the real Gemini, with your own key. Sections 1 to 3 use five made-up comments. From section 4 on, you work with your team on ten texts from your project data: you read them, write labels for them, label them yourselves, and compare your labels with Gemini's.

**Bring:** ten texts from your project data, saved as `lab07_team_texts.csv` in the same folder as this notebook, with two columns, `id` and `text`. Leave out texts with names, usernames, or other details that identify a person, because everything you send goes to Google.

## 1. Load your key

You need a Gemini API key in the `.env` file in the course repository's top folder. See "API keys" in `SETUP.md`. Keep the key in `.env`, which Git ignores. Do not paste it into this notebook or an AI chat.

Run the cell below. It should print `Key loaded.`

In [ ]:
from pathlib import Path
from datetime import date
import json
import logging
import os
import time

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

for folder in (Path.cwd(), *Path.cwd().parents):
    env_path = folder / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break

key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"

# Hide a warning that the google-genai package prints on every call. It does not mean anything went wrong.
logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Key loaded.")

## 2. One call

`client.models.generate_content` sends a prompt to a model. The answer is in `response.text`:

```python
response = client.models.generate_content(model=GEMINI_MODEL, contents="Your question here")
```

A response also counts the **tokens**: the pieces of words a model reads and writes. Services charge by the token. `response.usage_metadata.prompt_token_count` is the number of tokens you sent, and `response.usage_metadata.candidates_token_count` is the number in the answer.

Ask Gemini a short question and print `response.text`. Then print the two token counts.

If you get error 503, the server is busy. Wait a minute and run the cell again.

In [ ]:
# Send one question to Gemini and print response.text.


# Print the number of tokens sent and the number of tokens in the answer.



## 3. Label one comment

The cell below has five comments and a codebook, like the ones in the interactive, but about a different post. `SYSTEM` is the system instruction: the task and the definition of each label. `LABEL_SCHEMA` lists the fields the answer must have and the labels it may use. `CONFIG` puts both into the settings for a call.

In [ ]:
comments = pd.DataFrame(
    {
        "document_id": ["C1", "C2", "C3", "C4", "C5"],
        "text": [
            "You ate food with their name on it. You owe them an apology and a meal.",
            "Your roommate takes your things all the time. I don't see what you did wrong here.",
            "You shouldn't have eaten it, but shouting at you in front of guests was out of line too.",
            "Oh sure, a name on the box is just a suggestion. Great roommate behavior.",
            "Put a shelf in the fridge for each person and move on.",
        ],
    }
)

SYSTEM = """You label comments on a post. In the post, a person says they ate their roommate's leftovers, which had the roommate's name on the box.

Give the comment one label:
- poster_at_fault: the comment says the poster was in the wrong.
- poster_not_at_fault: the comment says the poster was not in the wrong.
- unclear: the comment does not say who was in the wrong.

Also give a reason of one short sentence."""

ALLOWED_LABELS = ["poster_at_fault", "poster_not_at_fault", "unclear"]

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": ALLOWED_LABELS},
        "reason": {"type": "string"},
    },
    "required": ["label", "reason"],
}

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    response_mime_type="application/json",
    response_schema=LABEL_SCHEMA,
)
comments

**Your turn:**

1. Write a function `build_prompt(text)` that returns the prompt for one comment. Use an f-string, such as `f"Comment: {text}"`. The task and the codebook are already in `SYSTEM`, so the prompt only needs to set out the comment.
2. Make `prompt = build_prompt(text=comments["text"][0])` and print it. This is what Gemini gets, together with `SYSTEM`.
3. Send the prompt with `config=CONFIG` and print `response.text`.
4. `response.text` is a string written as JSON. Turn it into a dictionary with `json.loads(response.text)`, and print the label.

In [ ]:
# Your turn
def build_prompt(text):
    return ...




## 4. Read your team's texts

The cell below reads `lab07_team_texts.csv`. If your team's file is not ready yet, it uses the five comments from section 3 instead, so you can still do every step.

In [ ]:
TEAM_PATH = Path("lab07_team_texts.csv")
if TEAM_PATH.exists():
    texts = pd.read_csv(TEAM_PATH, dtype={"id": str})
else:
    print("No team file yet. Using the five comments from section 3.")
    texts = comments.rename(columns={"document_id": "id"})

texts["n_words"] = texts["text"].str.split().str.len()
print(len(texts), "texts")
print(texts["n_words"].describe())
texts

**Your turn:** read all the texts together before you label anything. In the cell below, write down what you notice: how long the texts are, what language they are in, whether some repeat, and anything a word count would get wrong, such as links, emoji, or quotes from other texts. You will need this in Week 8.

In [ ]:
# What we notice in our texts:
# -

## 5. Write your labels

A label works best when you can point to the words in the text that support it. Choose two or three labels, plus `unclear`, and define each one in one sentence. Some examples:

- `mentions_price` / `no_price`
- `asks_for_help` / `gives_advice` / `unclear`
- `corrects_the_model` / `thanks_the_model` / `unclear`

If your data already has a label, such as a star rating or a flair, you can use its values as your labels.

**Your turn:** fill in `TEAM_SYSTEM` and `TEAM_LABELS`. The rest of the cell builds the schema and the settings, as in section 3.

In [ ]:
# Your turn: replace every ... and the label names.
TEAM_SYSTEM = """You label texts from ... . One text is ... .

Give the text one label:
- label_1: ...
- label_2: ...
- unclear: the text does not say.

Also give a reason of one short sentence."""

TEAM_LABELS = ["label_1", "label_2", "unclear"]

TEAM_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": TEAM_LABELS},
        "reason": {"type": "string"},
    },
    "required": ["label", "reason"],
}

TEAM_CONFIG = types.GenerateContentConfig(
    system_instruction=TEAM_SYSTEM,
    response_mime_type="application/json",
    response_schema=TEAM_SCHEMA,
)
assert "..." not in TEAM_SYSTEM, "Replace every ... in TEAM_SYSTEM."
print(TEAM_SYSTEM)

## 6. Label the texts yourselves

Two of you label every text, each without looking at the other's labels. Replace each list with your labels, one per text, in the same order as the table in section 4.

In [ ]:
texts["label_a"] = [""] * len(texts)  # first person's labels
texts["label_b"] = [""] * len(texts)  # second person's labels

assert texts["label_a"].isin(TEAM_LABELS).all(), "Every label in label_a must be one of TEAM_LABELS."
assert texts["label_b"].isin(TEAM_LABELS).all(), "Every label in label_b must be one of TEAM_LABELS."
texts[["id", "text", "label_a", "label_b"]]

## 7. Let Gemini label them

Some calls fail. Error 429 means too many calls in one minute, and error 503 means the server is busy. Both go away if you wait, so `ask` below waits and tries again, longer each time. Any other error, such as 404 for a misspelled model name, stops it right away, because trying again would give the same error. Read the code, then run the cell.

In [ ]:
RETRY_CODES = {429, 503}


def ask(prompt, config, max_tries=4):
    """Send one prompt with these settings. Return the answer as a dictionary. Try again after a temporary error."""
    wait = 10  # seconds to wait before the first retry
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config)
            time.sleep(6)  # wait between calls to stay under the free limit of calls per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in RETRY_CODES or attempt == max_tries:
                raise  # stop and show the error
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2  # wait longer each time: 10, 20, 40 seconds

**Your turn:** finish the loop below. For each text, build the prompt with `build_prompt(text=row.text)`, call `ask(prompt, TEAM_CONFIG)`, and add the answer to the file right away. If your texts are not comments, change the word `Comment` in `build_prompt` first.

```python
line = pd.DataFrame([{"id": row.id, "gemini_label": answer["label"], "reason": answer["reason"]}])
line.to_csv(GEMINI_PATH, mode="a", header=not GEMINI_PATH.exists(), index=False)
```

`mode="a"` adds a line to the end of the file instead of replacing it. Each answer is saved as soon as you get it, so if the loop stops, the answers you already have are safe. The loop skips texts that are already in the file.

Run the cell. Stop it after two or three texts with the stop button, then run it again. It should start where it stopped.

In [ ]:
GEMINI_PATH = Path("lab07_team_gemini.csv")
done = set(pd.read_csv(GEMINI_PATH, dtype={"id": str})["id"]) if GEMINI_PATH.exists() else set()

for row in texts.itertuples():
    if row.id in done:
        continue  # labeled in an earlier run
    # Your turn: build the prompt, call ask, and add one line to GEMINI_PATH.


print("Done.")

## 8. Compare the three sets of labels

**Your turn:**

1. Read the file into `gemini` with `dtype={"id": str}`, and merge it with `texts` on `id`.
2. Print three numbers: how often the two of you agree (hint: `(texts["label_a"] == texts["label_b"]).mean()`), how often person A agrees with Gemini, and how often person B does.
3. Make a table of `label_a` against `gemini_label` with `pd.crosstab`. It shows which labels get mixed up.
4. Read every text where any two of the three labels differ. For each one, write down which it was: one of you misread the text, Gemini got it wrong, or the definition is unclear.

In [ ]:
# Your turn


# Where we differ, and why:
# -

## 9. Change one definition

**Your turn:** copy `TEAM_SYSTEM` to `TEAM_SYSTEM_V2`, and change the definition that caused the most disagreement. Build `TEAM_CONFIG_V2` from it, the same way as `TEAM_CONFIG`. Run the loop from section 7 again, with `TEAM_CONFIG_V2` and a new file, `lab07_team_gemini_v2.csv`. Then print the texts whose Gemini label changed.

In [ ]:
# Your turn

## 10. Write down what ran

The same labels can come out differently with another model, another version of the definitions, or on another day. Make one table with `id`, `text`, `label_a`, `label_b`, and Gemini's labels from both versions. Add three columns: the model name (`GEMINI_MODEL`), the version of the definitions you used last, and today's date (`date.today().isoformat()`). Save it as `lab07_team_labels.csv`, and keep `TEAM_SYSTEM_V2` with it. You will use your labels again later in the course.

Last, write down one thing that labeling showed you about your data that matters for your project.

Check that your key does not appear in any cell or output. Then save the notebook.

In [ ]:
# Your turn


# What labeling showed us about our data:
# -